# 07 - K-Means Usage Clustering

I used clustering to group daily load shapes instead of only looking at one total number per day.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
hourly = pd.read_csv(
    "../01_Data/processed/hourly_energy.csv",
    parse_dates=["date_time"]
)

In [ ]:
profiles = hourly.pivot_table(
    index=hourly["date_time"].dt.date,
    columns="hour", values="energy_kwh", aggfunc="mean"
).dropna()

In [ ]:
profiles = profiles[profiles.columns[:24]]
print("Daily profiles:", len(profiles))

## Put days on the same scale

In [ ]:
scaled = profiles.div(profiles.mean(axis=1), axis=0)

## Try a few cluster counts

In [ ]:
for k in [2, 3, 4, 5]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(scaled)
    print(k, round(model.inertia_, 2))

## Use 3 clusters

In [ ]:
model = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = model.fit_predict(scaled)

In [ ]:
profiles["cluster"] = labels
print(profiles["cluster"].value_counts().sort_index())

## Plot the cluster shapes

In [ ]:
centers = pd.DataFrame(model.cluster_centers_, columns=range(24))

In [ ]:
plt.figure(figsize=(10,4))
for i in range(3):
    plt.plot(centers.loc[i], label=f"Cluster {i}")
plt.xlabel("Hour")
plt.ylabel("Relative load")
plt.title("Three Daily Energy Usage Patterns")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.savefig("../04_Visuals/clustering/11_cluster_profiles.png", dpi=150)
plt.show()


## Save the profiles

In [ ]:
profiles.index.name = "day"
profiles.reset_index().to_csv(
    "../01_Data/processed/daily_load_profiles.csv", index=False
)

The clusters are useful as simple usage patterns, such as steady use or stronger morning/evening use. I use them as a business view, not as fixed customer labels.